<div align="center">

# Air University, Islamabad
### Department of Computer Science · Faculty of Computing & Artificial Intelligence

---

## Assignment # 02 — Research Paper Reproduction

# Fine-Tuning Whisper for Pashto Automatic Speech Recognition Using Transfer Learning

*A reproduction of Rahman (2026), arXiv:2604.06507*

</div>

---

| | |
|:--|:--|
| **Course** | CS-444: Deep Learning |
| **Semester** | Fall 2026 |
| **Section** | BSCS-VII-B (Shift-II) |
| **Total Marks** | 50 |
| **Mapping** | CLO-2 · GA-3 · C3 (Apply) |
| **Instructor** | Mr. Qadeer Yasin |
| **Submission Date** | *&lt;fill in&gt;* |

### Group Members

| Name | Registration No. | Contribution |
|:--|:--|:--|
| *&lt;Member 1&gt;* | *&lt;reg no.&gt;* | Dataset acquisition and preprocessing |
| *&lt;Member 2&gt;* | *&lt;reg no.&gt;* | Implementation and experiment execution |
| *&lt;Member 3&gt;* | *&lt;reg no.&gt;* | Evaluation, metrics, figures and analysis |

*All members reviewed the full pipeline and can explain every number reported below.*

### Links

| | |
|:--|:--|
| **Code repository** | https://github.com/theawais77/pashto-whisper-asr-reproduction |
| **Base paper** | [arXiv:2604.06507](https://arxiv.org/abs/2604.06507) · DOI 10.48550/arXiv.2604.06507 |
| **Model checkpoint reproduced** | [ihanif/exp_009_small_cv24_vanilla](https://huggingface.co/ihanif/exp_009_small_cv24_vanilla) |
| **Dataset** | Mozilla Common Voice Scripted Speech 24.0 — Pashto (`ps`), CC0-1.0 |

---

## What this notebook does

The base paper fine-tunes OpenAI's Whisper model on Pashto, a language absent from Whisper's
pre-training data. Its recommended configuration (**exp_009**) reports a Word Error Rate of
**24.89 %** and a Character Error Rate of **8.075 %**, against a zero-shot baseline of
**120.4 % WER**.

This notebook reproduces that experiment and tests whether those published numbers hold up
when measured by an independently written evaluation harness. It covers every part of the
assignment brief:

| Brief | Where |
|:--|:--|
| (d) obtain, preprocess and split the dataset, documenting each step | Section 3 |
| (e) record the software environment, model settings and changes | Sections 1 and 2 |
| (f) apply suitable evaluation metrics and report results in a table | Sections 4, 6, 7, 9 |
| (g) visualise against the paper and compute the metric difference | Sections 9 and 10 |
| (h) limitations and whether the experiment is reproducible | Section 9 output, written up in the report |

The original authors did not publish their training code, so the implementation below is our own,
written against the public `transformers` API.

---



# Batch 3 (Kaggle) — reproduction of exp_009

Reproduction of **Rahman (2026), arXiv:2604.06507** — exp_009: whisper-small on Common Voice v24 Pashto, reported **24.89 % WER / 8.075 % CER**, zero-shot baseline **120.4 % WER**.

Kaggle-only, Route B only. No training. Run top to bottom.

**Before you start — Settings panel, right side:**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (separate toggle from GPU)
- Add Input → Datasets → `ataullahaali/common-voice-scripted-speech-24-0-pashto`

| Cell | Does | Time |
|---|---|---|
| 1 | Environment check, installs only what is missing | 1 min |
| 2 | Recovers the author's unpublished config + his training curve | 1 min |
| 3 | Builds the test split | 2 min |
| 4–5 | Metrics and decoding | instant |
| 6 | **B0** zero-shot baseline | ~5 min |
| 7 | **B** author's checkpoint — the reproduction | ~4 min |
| 8 | Normalisation ablation (ours) | seconds |
| 9 | Results table and the §7.1 difference block | instant |
| 10 | Figures | instant |

**Cell 2 needs no GPU and no dataset.** If everything else fails, its output is still genuine execution evidence — screenshot it.

## 1 · Environment

In [ ]:
import os, sys, glob, subprocess, importlib, pathlib

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

# internet (separate Kaggle toggle from GPU)
try:
    import urllib.request
    urllib.request.urlopen("https://huggingface.co", timeout=10)
    print("internet  : ON")
except Exception as e:
    raise SystemExit(f"internet OFF -> Settings panel -> Internet -> On. ({e})")

# install ONLY what is missing; Kaggle preinstalls most of it
missing = []
for m in ["torch", "transformers", "evaluate", "jiwer", "librosa",
          "soundfile", "pandas", "numpy", "matplotlib"]:
    try:
        importlib.import_module(m)
    except ImportError:
        missing.append(m)
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all packages present")

import torch, transformers, evaluate, jiwer
print("\ngpu       :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

env = "\n".join([f"python        : {sys.version.split()[0]}",
                 f"torch         : {torch.__version__}",
                 f"cuda          : {torch.version.cuda}",
                 f"gpu           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}",
                 f"transformers  : {transformers.__version__}",
                 f"evaluate      : {evaluate.__version__}",
                 f"jiwer         : {jiwer.__version__}"])
print(env)
(RESULTS / "environment.txt").write_text(env, encoding="utf-8")

# dataset mount
hits = glob.glob("/kaggle/input/**/test.tsv", recursive=True)
if not hits:
    print("\n/kaggle/input contains:", glob.glob("/kaggle/input/*"))
    raise SystemExit("No test.tsv. Add Input -> Datasets -> "
                     "ataullahaali/common-voice-scripted-speech-24-0-pashto")
CV_ROOT = pathlib.Path(hits[0]).parent
print("\nCommon Voice root:", CV_ROOT)
print("contents:", sorted(p.name for p in CV_ROOT.iterdir())[:12])
assert (CV_ROOT / "clips").exists(), f"no clips/ folder in {CV_ROOT}"

## 2 · Recover the author's unpublished configuration

The paper never states its optimizer, warmup, precision or batch composition. But the author's Hugging Face checkpoint ships `experiment_log.txt` — the full training log of exp_009 — which is public and ungated.

This recovers values the paper omits, **and one finding worth reporting**: `eval_dataset_splits: ['test']`. The evaluation loop that selected his best checkpoint ran on the **test** split, so the reported 24.89 % is selected on the same data it is reported on. The paper calls this "validation WER"; no separate validation set existed.

Needs no GPU and no dataset.

In [ ]:
import re, csv, urllib.request, pathlib, json
RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)
CKPT = "ihanif/exp_009_small_cv24_vanilla"

urllib.request.urlretrieve(
    f"https://huggingface.co/{CKPT}/resolve/main/experiment_log.txt",
    RESULTS / "experiment_log.txt")
log = (RESULTS / "experiment_log.txt").read_text(encoding="utf-8", errors="replace")

args = dict(re.findall(r"^([a-z_]+)\s+:\s(.*)$", log.split("=" * 70)[2], flags=re.M))
json.dump(args, open(RESULTS / "author_exp009_args.json", "w", encoding="utf-8"), indent=2)

print("=" * 72)
print("CONFIGURATION OF exp_009, RECOVERED FROM THE AUTHOR'S PUBLISHED LOG")
print("=" * 72)
for k in ["model_name", "train_dataset_splits", "eval_dataset_splits", "learning_rate",
          "train_batchsize", "gradient_accumulation_steps", "warmup_steps", "warmup_ratio",
          "weight_decay", "fp16", "bf16", "num_epochs", "eval_steps",
          "early_stopping_patience", "metric_for_best_model", "gradient_checkpointing",
          "min_duration_in_seconds", "max_duration_in_seconds", "use_lora",
          "freeze_encoder_layers", "augment_speed", "augment_noise"]:
    if k in args:
        print(f"  {k:30s} = {args[k]}")
print("  optim                          = not passed -> Seq2SeqTrainer default (adamw_torch)")
print("\n*** eval_dataset_splits = test : checkpoint selection ran on the TEST split ***")

In [ ]:
rows = [dict(re.findall(r"'([a-z_]+)':\s*'([^']*)'", m.group(0)))
        for m in re.finditer(r"\{'eval_loss':.*?\}", log)]
keys = ["epoch", "eval_loss", "eval_wer", "eval_wer_ortho", "eval_cer"]
with open(RESULTS / "paper_exp009_eval_curve.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=keys); w.writeheader()
    for r in rows:
        w.writerow({k: r.get(k, "") for k in keys})

print(f"{len(rows)} evaluation points recovered\n")
print(f"{'epoch':>7}{'loss':>9}{'WER':>8}{'ortho':>8}{'CER':>8}")
for r in rows:
    print(f"{float(r['epoch']):7.3f}{float(r['eval_loss']):9.4f}"
          f"{float(r['eval_wer']):8.2f}{float(r['eval_wer_ortho']):8.2f}{float(r['eval_cer']):8.3f}")

best = min(rows, key=lambda r: float(r["eval_wer"]))
lo   = min(rows, key=lambda r: float(r["eval_loss"]))
print(f"\nbest WER    {best['eval_wer']}% at epoch {best['epoch']}   <- the paper's headline 24.89%")
print(f"lowest loss {lo['eval_loss']} at epoch {lo['epoch']} (WER {lo['eval_wer']}%)")
print("Loss and WER bottom at DIFFERENT epochs -> that is why checkpoints are")
print("selected on WER, not on loss. Worth a sentence in section 9.")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
e = [float(r["epoch"]) for r in rows]
wv = [float(r["eval_wer"]) for r in rows]
lv = [float(r["eval_loss"]) for r in rows]
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(e, wv, marker="o", ms=5, lw=1.5, color="#c53030", label="eval WER")
ax.scatter([float(best["epoch"])], [float(best["eval_wer"])], marker="*", s=280,
           color="#742a2a", zorder=5,
           label=f"best {best['eval_wer']}% @ epoch {best['epoch']}")
ax.set_xlabel("Epoch"); ax.set_ylabel("WER (%)", color="#c53030")
ax.tick_params(axis="y", colors="#c53030")
tw = ax.twinx()
tw.plot(e, lv, marker="s", ms=3, lw=1.2, color="#4a5568", alpha=0.75, label="eval loss")
tw.set_ylabel("Validation loss", color="#4a5568")
tw.axvline(float(lo["epoch"]), ls=":", color="#4a5568", lw=1)
h = ax.get_lines() + tw.get_lines()
ax.legend(h, [x.get_label() for x in h], fontsize=8, loc="upper right")
ax.set_title("The paper's own exp_009 training curve,\nreconstructed from its published log")
ax.grid(alpha=0.3)
for ext in ("png", "pdf"):
    fig.savefig(RESULTS / f"paper_exp009_curve_reproduced.{ext}", dpi=300, bbox_inches="tight")
plt.close(fig)
print("\nwrote results/paper_exp009_curve_reproduced.png")

## 3 · Build the test split

Official shipped `test.tsv` only — no re-splitting, because Common Voice builds its splits speaker-disjoint and a custom split would leak speakers and inflate the score.

Filters use the author's **recovered** duration window (0.5–30 s), not a guess.

`MAX_EVAL_SAMPLES` is a seeded subsample. **Whatever you set, §4.2 of the report must say so** — declared sampling is fine, undeclared is not.

In [ ]:
MAX_EVAL_SAMPLES = 400     # 400 ~= 6 min/system. None = full 13,791 (~2.5 h/system)
SEED = 42

PAPER = {"wer": 24.89, "wer_ortho": 27.73, "cer": 8.075,
         "zeroshot_wer": 120.4, "test_rows": 13791, "test_hours": 20.72}

import csv as _csv, hashlib, json
import pandas as pd, numpy as np

df = pd.read_csv(CV_ROOT / "test.tsv", sep="\t", quoting=_csv.QUOTE_NONE,
                 dtype=str, keep_default_na=False)
n_raw = len(df)
print(f"official test.tsv rows: {n_raw}   (paper: {PAPER['test_rows']})")

df["audio"] = [str(CV_ROOT / "clips" / p) for p in df["path"]]

dur_tsv = CV_ROOT / "clip_durations.tsv"
if dur_tsv.exists():
    d = pd.read_csv(dur_tsv, sep="\t")
    durs = dict(zip(d[d.columns[0]], d[d.columns[1]] / 1000.0))
    df["duration_s"] = [durs.get(p, np.nan) for p in df["path"]]
else:
    df["duration_s"] = np.nan

steps = []
def drop(frame, rule, mask):
    before = len(frame); frame = frame[mask].copy()
    steps.append({"rule": rule, "rows_in": before, "rows_out": len(frame),
                  "removed": before - len(frame)})
    print(f"  {rule:42s} {before:>6} -> {len(frame):>6}")
    return frame

print("filters:")
df = drop(df, "drop empty transcripts", df["sentence"].str.strip() != "")
if df["duration_s"].notna().any():
    df = drop(df, "duration in [0.5, 30.0] s (recovered config)",
              df["duration_s"].between(0.5, 30.0, inclusive="both"))

n_full = len(df)
hrs = df["duration_s"].sum() / 3600.0 if df["duration_s"].notna().any() else float("nan")
print(f"\nafter filters: {n_full} utterances" + (f" / {hrs:.2f} h" if hrs == hrs else ""))

if MAX_EVAL_SAMPLES and MAX_EVAL_SAMPLES < len(df):
    df = df.sample(n=MAX_EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)
    print(f"*** SEEDED SUBSAMPLE: {len(df)} of {n_full} -- state this in report section 4.2 ***")
else:
    df = df.reset_index(drop=True)
    print("using the FULL official test split")

fingerprint = hashlib.sha256("|".join(map(str, df["sentence"])).encode()).hexdigest()[:16]
print(f"\nTEST SPLIT FINGERPRINT: {fingerprint}")
if "client_id" in df:
    print(f"unique speakers: {df['client_id'].nunique()}")

pd.DataFrame(steps).to_csv(RESULTS / "preprocessing_log.csv", index=False)
json.dump({"rows_raw": int(n_raw), "rows_after_filters": int(n_full),
           "rows_evaluated": int(len(df)),
           "hours_after_filters": None if hrs != hrs else round(float(hrs), 3),
           "speakers": int(df["client_id"].nunique()) if "client_id" in df else None,
           "seed": SEED, "subsample": MAX_EVAL_SAMPLES, "fingerprint": fingerprint,
           "paper": PAPER},
          open(RESULTS / "dataset_stats.json", "w", encoding="utf-8"), indent=2)
TEST = df
REFS = list(TEST["sentence"])
print("wrote results/preprocessing_log.csv and results/dataset_stats.json")

## 4 · Metrics

The paper's headline metric is **normalised WER** (both sides through HuggingFace `BasicTextNormalizer`), so that is what the §7.1 difference uses on both sides. We also report orthographic WER and CER.

`BasicTextNormalizer` strips punctuation but performs **no Arabic-script canonicalisation** — it will not unify Arabic yeh `ي` with Pashto yeh `ی`, nor Arabic kaf `ك` with Pashto kaf `ک`, and leaves ZWNJ and tatweel in place. Cell 8 measures how much that is worth. That ablation is ours; it is not in the paper.

In [ ]:
import unicodedata, numpy as np, evaluate as hf_evaluate, jiwer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

_norm = BasicTextNormalizer()
_wer, _cer = hf_evaluate.load("wer"), hf_evaluate.load("cer")

PASHTO_DISTINCTIVE = set("ټډړڼږښځڅګۍ")
_CANON = {"ي": "ی", "ى": "ی", "ك": "ک",
          "ۀ": "ه", "ة": "ه"}
_STRIP = ("‌‍​ـًٌٍَُِّ"
          "ْٰ،؛؟۔")
_DIGITS = {**{chr(0x0660 + i): str(i) for i in range(10)},
           **{chr(0x06F0 + i): str(i) for i in range(10)}}

def pashto_canonical(t):
    for a, b in _CANON.items():
        t = t.replace(a, b)
    t = "".join(_DIGITS.get(c, c) for c in t if c not in _STRIP)
    return " ".join(unicodedata.normalize("NFC", t).split())

def score(refs, hyps, extra=None):
    rn, hn, ro, ho = [], [], [], []
    for r, h in zip(refs, hyps):
        if extra:
            r, h = extra(r), extra(h)
        a, b = _norm(r).strip(), _norm(h).strip()
        if not a:
            continue
        rn.append(a); hn.append(b or " "); ro.append(r); ho.append(h or " ")
    o = jiwer.process_words(rn, hn)
    err = o.substitutions + o.deletions + o.insertions
    return {"wer": 100 * _wer.compute(references=rn, predictions=hn),
            "wer_ortho": 100 * _wer.compute(references=ro, predictions=ho),
            "cer": 100 * _cer.compute(references=rn, predictions=hn),
            "n": len(rn), "sub": o.substitutions, "del": o.deletions, "ins": o.insertions,
            "sub_pct": round(100 * o.substitutions / err, 2) if err else 0,
            "del_pct": round(100 * o.deletions / err, 2) if err else 0,
            "ins_pct": round(100 * o.insertions / err, 2) if err else 0,
            "pashto_script_rate": 100 * float(np.mean(
                [any(c in PASHTO_DISTINCTIVE for c in h) for h in hyps]))}

def bootstrap_ci(refs, hyps, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed); c = []
    for r, h in zip(refs, hyps):
        a = _norm(r).strip()
        if not a:
            continue
        o = jiwer.process_words(a, _norm(h).strip() or " ")
        c.append((o.substitutions + o.deletions + o.insertions, len(a.split())))
    c = np.array(c, dtype=float)
    idx = rng.integers(0, len(c), size=(n_boot, len(c)))
    d = 100 * c[idx, 0].sum(axis=1) / c[idx, 1].sum(axis=1)
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

# sanity: an empty hypothesis must score exactly 100% WER (all deletions)
chk = score(["سلام وروره"], [""])
print(f"sanity check -- empty hypothesis WER = {chk['wer']:.1f}% (must be 100.0)")

## 5 · Decoding

One function for every system, so the comparison is valid. Greedy, `max_new_tokens=225`, `language="ps"` forced. **Do not change these between cells 6 and 7** or the comparison is void.

In [ ]:
import torch, time, gc, librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
BATCH, NUM_BEAMS, MAX_NEW_TOKENS = 16, 1, 225

def transcribe_all(model_id, frame, tag):
    print(f"\n=== {tag}  ({model_id})")
    proc = WhisperProcessor.from_pretrained(model_id, language="ps", task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=DTYPE, low_cpu_mem_usage=True).to(DEVICE).eval()
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    print(f"    parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

    hyps, t0 = [], time.time()
    for s in range(0, len(frame), BATCH):
        chunk = frame.iloc[s:s + BATCH]
        audio = [librosa.load(p, sr=16000, mono=True)[0] for p in chunk["audio"]]
        feats = proc.feature_extractor(audio, sampling_rate=16000,
                                       return_tensors="pt").input_features.to(DEVICE, DTYPE)
        with torch.no_grad():
            ids = model.generate(feats, language="ps", task="transcribe",
                                 num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS)
        hyps.extend(proc.tokenizer.batch_decode(ids, skip_special_tokens=True))
        done = s + len(chunk)
        if s % (BATCH * 5) == 0 or done == len(frame):
            el = time.time() - t0
            print(f"    {done:>5}/{len(frame)}  {el:>5.0f}s  "
                  f"eta {el/max(done,1)*(len(frame)-done):>5.0f}s", flush=True)

    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"    done in {time.time()-t0:.0f}s")
    return [h.strip() for h in hyps]

print(f"device={DEVICE} dtype={DTYPE} batch={BATCH} greedy max_new_tokens={MAX_NEW_TOKENS}")

## 6 · B0 — zero-shot baseline

Untouched `openai/whisper-small`. Paper reports **120.4 % WER**.

Above 100 % is expected: insertions are unbounded while the reference length is fixed, so a model generating fluent text in the *wrong script* accumulates insertions without limit. A system outputting nothing scores exactly 100 %, so zero-shot Whisper is worse than silence on Pashto.

In [ ]:
import pandas as pd
hyp_zs = transcribe_all("openai/whisper-small", TEST, "B0 zero-shot whisper-small")
m_zs   = score(REFS, hyp_zs)
ci_zs  = bootstrap_ci(REFS, hyp_zs)

print(f"\n{'':<20}{'ours':>9}{'paper':>9}{'diff pp':>10}")
print(f"{'WER (normalised)':<20}{m_zs['wer']:>9.2f}{PAPER['zeroshot_wer']:>9.2f}"
      f"{m_zs['wer']-PAPER['zeroshot_wer']:>+10.2f}")
print(f"{'WER (orthographic)':<20}{m_zs['wer_ortho']:>9.2f}{'n/a':>9}")
print(f"{'CER':<20}{m_zs['cer']:>9.2f}{'n/a':>9}")
print(f"\n95% bootstrap CI: {ci_zs[0]:.2f} - {ci_zs[1]:.2f}")
print(f"errors: S {m_zs['sub']} / D {m_zs['del']} / I {m_zs['ins']}  "
      f"-> insertions {m_zs['ins_pct']}% of all errors")
print(f"Pashto-script output rate: {m_zs['pashto_script_rate']:.2f}%  "
      f"(companion paper arXiv:2604.04598 reports <0.8%)")

pd.DataFrame({"reference": REFS, "hypothesis": hyp_zs}).to_csv(
    RESULTS / "predictions_zeroshot.csv", index=False, encoding="utf-8")
print("\nfirst two outputs -- note the script they are in:")
for r, h in list(zip(REFS, hyp_zs))[:2]:
    print(f"  REF: {r}\n  HYP: {h[:140]}\n")

## 7 · B — the reproduction

The author's released checkpoint, scored by **our** harness. Paper reports **24.89 % WER / 8.075 % CER**.

This is the cell that answers *"is the paper's reported number verifiable?"* — a different question from *"does its training recipe reproduce?"*, which needs Batch 4.

In [ ]:
hyp_ck = transcribe_all(CKPT, TEST, "B author's exp_009 checkpoint")
m_ck   = score(REFS, hyp_ck)
ci_ck  = bootstrap_ci(REFS, hyp_ck)

print(f"\n{'':<20}{'ours':>9}{'paper':>9}{'diff pp':>10}{'rel %':>9}")
for lab, k, ref in (("WER (normalised)", "wer", PAPER["wer"]),
                    ("WER (orthographic)", "wer_ortho", PAPER["wer_ortho"]),
                    ("CER", "cer", PAPER["cer"])):
    d = m_ck[k] - ref
    print(f"{lab:<20}{m_ck[k]:>9.2f}{ref:>9.3f}{d:>+10.2f}{100*d/ref:>+9.1f}")

print(f"\n95% bootstrap CI on WER: {ci_ck[0]:.2f} - {ci_ck[1]:.2f}")
inside = ci_ck[0] <= PAPER["wer"] <= ci_ck[1]
print(f"the paper's 24.89% lies {'INSIDE' if inside else 'OUTSIDE'} our CI -> "
      f"{'REPRODUCED' if inside else 'discrepancy, explain it in section 9'}")
print(f"errors: S {m_ck['sub']} / D {m_ck['del']} / I {m_ck['ins']}  "
      f"-> substitutions {m_ck['sub_pct']}% (paper reports ~70-78%)")
print(f"Pashto-script rate: {m_ck['pashto_script_rate']:.2f}% "
      f"(zero-shot was {m_zs['pashto_script_rate']:.2f}%)")
print(f"WER/CER ratio: {m_ck['wer']/m_ck['cer']:.2f}  (paper {PAPER['wer']/PAPER['cer']:.2f})")

pd.DataFrame({"reference": REFS, "hypothesis": hyp_ck}).to_csv(
    RESULTS / "predictions_author_checkpoint.csv", index=False, encoding="utf-8")

## 8 · D — orthographic-normalisation ablation (ours)

Re-scores the **same** predictions under Pashto canonicalisation. No re-decoding, so it costs seconds. Whatever drops out was Arabic-script spelling variation rather than recognition error. Not comparable to the paper's 24.89 % — say so in the caption.

In [ ]:
rows = []
for tag, hyp in (("zero-shot", hyp_zs), ("author exp_009 checkpoint", hyp_ck)):
    a, b = score(REFS, hyp), score(REFS, hyp, extra=pashto_canonical)
    rows.append({"system": tag,
                 "wer_paper_normaliser": round(a["wer"], 2),
                 "wer_plus_pashto_canon": round(b["wer"], 2),
                 "wer_reduction_pp": round(a["wer"] - b["wer"], 2),
                 "cer_paper_normaliser": round(a["cer"], 2),
                 "cer_plus_pashto_canon": round(b["cer"], 2)})
abl = pd.DataFrame(rows)
abl.to_csv(RESULTS / "normalisation_ablation.csv", index=False, encoding="utf-8")
print(abl.to_string(index=False))
print(f"\nOn the author's checkpoint, {rows[1]['wer_reduction_pp']} pp of the measured WER")
print("is Pashto spelling variation rather than recognition error.")

## 9 · Results table and the §7.1 block

Copy the printed block straight into the report.

In [ ]:
import torch
metrics = pd.DataFrame([
    {"tag": "paper_zeroshot", "source": "paper", "experiment": "zero-shot",
     "model": "openai/whisper-small", "wer": PAPER["zeroshot_wer"]},
    {"tag": "paper_exp009", "source": "paper", "experiment": "fine-tuned (paper)",
     "model": CKPT, "wer": PAPER["wer"], "wer_ortho": PAPER["wer_ortho"], "cer": PAPER["cer"]},
    {"tag": "zeroshot", "source": "ours", "experiment": "zero-shot",
     "model": "openai/whisper-small", "n_utterances": m_zs["n"],
     "wer": round(m_zs["wer"], 2), "wer_ortho": round(m_zs["wer_ortho"], 2),
     "cer": round(m_zs["cer"], 2), "wer_ci_lo": round(ci_zs[0], 2),
     "wer_ci_hi": round(ci_zs[1], 2), "paper_wer": PAPER["zeroshot_wer"],
     "difference_pp": round(m_zs["wer"] - PAPER["zeroshot_wer"], 2),
     "ins_pct_of_errors": m_zs["ins_pct"],
     "pashto_script_rate": round(m_zs["pashto_script_rate"], 2)},
    {"tag": "author_checkpoint", "source": "ours", "experiment": "author checkpoint",
     "model": CKPT, "n_utterances": m_ck["n"],
     "wer": round(m_ck["wer"], 2), "wer_ortho": round(m_ck["wer_ortho"], 2),
     "cer": round(m_ck["cer"], 2), "wer_ci_lo": round(ci_ck[0], 2),
     "wer_ci_hi": round(ci_ck[1], 2), "paper_wer": PAPER["wer"], "paper_cer": PAPER["cer"],
     "difference_pp": round(m_ck["wer"] - PAPER["wer"], 2),
     "difference_rel_pct": round(100*(m_ck["wer"]-PAPER["wer"])/PAPER["wer"], 1),
     "sub_pct_of_errors": m_ck["sub_pct"],
     "pashto_script_rate": round(m_ck["pashto_script_rate"], 2)},
])
for d in metrics.to_dict("records"):
    d.setdefault("num_beams", NUM_BEAMS)
metrics["num_beams"] = NUM_BEAMS
metrics["eval_subsample"] = MAX_EVAL_SAMPLES or "full split"
metrics["seed"] = SEED
metrics["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
metrics.to_csv(RESULTS / "metrics.csv", index=False, encoding="utf-8")

print("=" * 76)
print("SECTION 7.1 -- DIFFERENCE IN THE MAIN EVALUATION METRIC")
print("=" * 76)
print("Main metric: normalised WER. Sign: difference = ours - paper (positive = worse).\n")
print("Part B  (author's released checkpoint, our harness)")
print(f"  WER_paper           = {PAPER['wer']:.2f} %")
print(f"  WER_ours            = {m_ck['wer']:.2f} %   95% CI [{ci_ck[0]:.2f}, {ci_ck[1]:.2f}]")
print(f"  Absolute difference = {m_ck['wer']-PAPER['wer']:+.2f} pp")
print(f"  Relative difference = {100*(m_ck['wer']-PAPER['wer'])/PAPER['wer']:+.1f} %")
print(f"  CER_paper = {PAPER['cer']:.3f} %   CER_ours = {m_ck['cer']:.2f} %   "
      f"difference = {m_ck['cer']-PAPER['cer']:+.2f} pp")
print("\nZero-shot baseline")
print(f"  WER_paper = {PAPER['zeroshot_wer']:.2f} %   WER_ours = {m_zs['wer']:.2f} %   "
      f"difference = {m_zs['wer']-PAPER['zeroshot_wer']:+.2f} pp")
print("\nScript-failure analysis (ours, not in the paper)")
print(f"  zero-shot Pashto-script output rate  = {m_zs['pashto_script_rate']:.2f} %")
print(f"  checkpoint Pashto-script output rate = {m_ck['pashto_script_rate']:.2f} %")
print(f"  insertions as a share of zero-shot errors = {m_zs['ins_pct']:.1f} %")
print("=" * 76)
print(f"\nEvaluated on {m_ck['n']} utterances"
      f"{' (seeded subsample -- state this in section 4.2)' if MAX_EVAL_SAMPLES else ' (full split)'}")
metrics

## 10 · Figures

Paper bars hatched and muted, ours solid — the two sources stay distinguishable in the charts, not only in the prose. Both panels start at zero so neither gap is exaggerated.

In [ ]:
import matplotlib.pyplot as plt
PAPER_STYLE = dict(color="#9aa5b1", hatch="//", edgecolor="#4a5568", linewidth=1.2)
OURS_STYLE  = dict(color="#2b6cb0", edgecolor="#1a365d", linewidth=1.2)

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(RESULTS / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig); print("  ", name)

print("figures:")

# zero-shot vs fine-tuned
fig, ax = plt.subplots(figsize=(5.5, 4.4))
v = [m_zs["wer"], m_ck["wer"]]
b = ax.bar(["Zero-shot", "Author checkpoint"], v, width=0.55, **OURS_STYLE)
b[0].set_color("#c53030"); b[0].set_edgecolor("#742a2a")
for r in b:
    ax.annotate(f"{r.get_height():.2f}%", (r.get_x()+r.get_width()/2, r.get_height()),
                ha="center", va="bottom", fontsize=10)
ax.axhline(100, ls=":", color="#718096", lw=1.2)
ax.text(1.45, 101.5, "100% = output nothing", fontsize=7.5, color="#718096", ha="right")
ax.set_ylabel("WER (%)"); ax.set_ylim(0, max(v)*1.2)
ax.set_title("Pashto WER: zero-shot vs fine-tuned whisper-small")
ax.grid(axis="y", alpha=0.3)
save(fig, "wer_zeroshot_vs_finetuned")

# ours vs paper, two panels
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, (title, th, ou, ci) in zip(axes, [
        ("Zero-shot whisper-small", PAPER["zeroshot_wer"], m_zs["wer"], ci_zs),
        ("Author's exp_009 checkpoint", PAPER["wer"], m_ck["wer"], ci_ck)]):
    ax.bar([0], [th], 0.5, label="Rahman (2026), reported", **PAPER_STYLE)
    ax.bar([1], [ou], 0.5, label="This reproduction", **OURS_STYLE)
    ax.errorbar([1], [ou], yerr=[[ou-ci[0]], [ci[1]-ou]], fmt="none",
                ecolor="#1a365d", capsize=5, lw=1.4)
    ax.annotate(f"{th:.2f}", (0, th), ha="center", va="bottom", fontsize=10,
                xytext=(0, 3), textcoords="offset points")
    ax.annotate(f"{ou:.2f}", (1, ci[1]), ha="center", va="bottom", fontsize=10,
                xytext=(0, 4), textcoords="offset points")
    top = max(th, ou, ci[1]) * 1.38
    ax.set_ylim(0, top)
    ax.annotate(f"{ou-th:+.2f} pp\n({100*(ou-th)/th:+.1f} %)", (0.5, top*0.88),
                ha="center", va="top", fontsize=10, color="#742a2a", weight="bold")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["paper", "ours"]); ax.set_xlim(-0.6, 1.6)
    ax.set_title(title, fontsize=11); ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("WER (%)")
axes[1].legend(frameon=False, fontsize=9, loc="upper right")
fig.suptitle(f"Our reproduction against the paper's reported WER  (N={m_ck['n']})", y=1.04)
save(fig, "wer_comparison")

# error breakdown
fig, ax = plt.subplots(figsize=(6, 4.4))
labels = ["Zero-shot", "Author checkpoint"]
series = {"Substitutions": [m_zs["sub_pct"], m_ck["sub_pct"]],
          "Deletions":     [m_zs["del_pct"], m_ck["del_pct"]],
          "Insertions":    [m_zs["ins_pct"], m_ck["ins_pct"]]}
colours = {"Substitutions": "#2b6cb0", "Deletions": "#d69e2e", "Insertions": "#c53030"}
bottom = [0, 0]
for k, vals in series.items():
    ax.bar(labels, vals, bottom=bottom, label=k, color=colours[k],
           edgecolor="white", width=0.5)
    bottom = [x+y for x, y in zip(bottom, vals)]
ax.axhline(74, ls="--", color="#4a5568", lw=1)
ax.text(1.45, 75.5, "paper: ~70-78% substitutions", fontsize=7.5, ha="right", color="#4a5568")
ax.set_ylabel("Share of total word errors (%)"); ax.set_ylim(0, 105)
ax.set_title("Error-type breakdown"); ax.legend(frameon=False, ncol=3, fontsize=8)
ax.grid(axis="y", alpha=0.3)
save(fig, "error_breakdown")

print("\nfiles in results/:")
for p in sorted(RESULTS.iterdir()):
    print(f"  {p.stat().st_size:>10,}  {p.name}")
print("\nDownload them from the Output tab on the right.")